# Minerva: prepared protein/DNA sequences

![Minerva](https://proto-bio.github.io/proto-assets/images/tool/minerva/hero.png)

This notebook tours every Minerva tool: embeddings, masked pseudo-log-likelihood, modality-preserving sampling, and relaxed-sequence gradients, plus interaction maps. See [the Minerva software repository](https://github.com/garykbrixi/minerva) for the upstream model. Code and released weights use Apache-2.0.

Inputs are already prepared: uppercase protein, lowercase DNA, and `+` / `-` strand markers. The upstream `<+>` / `<->` spelling is also accepted and converted to `+` / `-`. The tool preserves case and orientation. It does not call genes or translate a genome. Token positions equal 1-indexed string positions.

The cells build an isolated model environment and download the checkpoint on first execution. A CUDA GPU is selected explicitly. Scoring and gradients use a short fragment to keep the example bounded.

In [1]:
from pathlib import Path
from tempfile import mkdtemp

from proto_tools.tools.masked_models.minerva import (
    MinervaEmbeddingsConfig, MinervaEmbeddingsInput, run_minerva_embeddings,
    MinervaScoringConfig, MinervaScoringInput, run_minerva_score,
    MinervaSampleConfig, MinervaSampleInput, run_minerva_sample,
    MinervaGradientConfig, MinervaGradientInput, run_minerva_gradient,
    MinervaInteractionsConfig, MinervaInteractionsInput, run_minerva_interactions,
)
from proto_tools.tools.masked_models.mixed_data_models import one_hot_mixed_logits
from proto_tools.transforms.masking import MaskingStrategy
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools

CHECKPOINT = "gbrixi/minerva-mlm"
DEVICE = "cuda"
OUTPUT_DIR = Path(mkdtemp(prefix="proto_minerva_example_"))

display_available_tools("minerva")
print("Exports:", OUTPUT_DIR)

- **`run_minerva_embeddings()`** — Extract mixed protein/DNA embeddings and biological logits with Minerva
- **`run_minerva_gradient()`** — Differentiate masked pseudo-log-likelihood for relaxed mixed protein/DNA tokens with Minerva
- **`run_minerva_interactions()`** — Predict protein, RNA base-pairing, and repeat interaction maps with Minerva
- **`run_minerva_sample()`** — Sample mixed protein/DNA loci while preserving modality and strand markers with Minerva
- **`run_minerva_score()`** — Score mixed protein/DNA loci by masked pseudo-log-likelihood with Minerva

Exports: /tmp/proto_minerva_example_u0or767v


## Prepared example locus

The protein and DNA segments below are shortened from the mixed-locus example in the [gLM2 upstream README](https://github.com/TattaBio/gLM2#usage). This illustrates the input notation, not a reconstruction with genomic coordinates. Shortening the context also changes model predictions.

The smaller fragment is used for the more expensive one-mask-per-position score and gradient calculations.

In [2]:
SEQUENCE = "+MALTKVEKRNRIK+aatttaaggaa-MLGIDNIERVKP"
SHORT_SEQUENCE = "+MALTKV+aatt-MLGIDN"
print(SEQUENCE)

+MALTKVEKRNRIK+aatttaaggaa-MLGIDNIERVKP


## Embeddings and token labels

The pooled vector averages all unpadded tokens, including strand markers. Optional raw logits have one row per token and 24 columns in `ACDEFGHIKLMNPQRSTVWYacgt` order.

In [3]:
display_api_reference("minerva-embedding", "input", "run_minerva_embeddings")
display_api_reference("minerva-embedding", "config", "run_minerva_embeddings")
display_api_reference("minerva-embedding", "output", "run_minerva_embeddings")

**Input** — `MixedSequenceInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Prepared loci: uppercase protein, lowercase acgt, +/- strand markers (<+>/<-> accepted) |

**Config** — `MinervaEmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device used for model inference |
| <code>model_checkpoint</code> | <code>Literal['gbrixi/minerva-mlm', 'gbrixi/minerva-mlm-8k']</code> | <code>'gbrixi/minerva-mlm'</code> | Minerva checkpoint; downloaded automatically from Hugging Face |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences or masked PLL variants per forward pass; reduce if memory is limited |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include token-aligned logits over the 24 canonical biological tokens |
| <code>repr_layer</code> | <code>int</code> | <code>-1</code> | 0=embedding table, 1..N=transformer outputs, -1=last transformer output |

**Output** — `MixedEmbeddingsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[MixedSequenceEmbedding]</code> | required | Embedding bundles in input order |

**`MixedSequenceEmbedding`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>mean_embedding</code> | <code>list[float]</code> | required | Mean-pooled embedding vector (averaged over sequence length) |
| <code>attention_mask</code> | <code>list[int]</code> | required | Binary mask: 1 = valid position, 0 = padding |
| <code>logits</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Optional per-token biological logits in vocabulary order |
| <code>projection</code> | <code>Projection2D &#124; None</code> | <code>None</code> | 2D UMAP projection of this sequence's embedding within the call's batch |
| <code>tokens</code> | <code>list[str]</code> | required | Unpadded token axis, including strand markers |
| <code>vocab</code> | <code>list[str]</code> | <code>['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'a', 'c', 'g', 't']</code> | Biological-logit column order |

**`Projection2D`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>x</code> | <code>float</code> | required | First reduced coordinate |
| <code>y</code> | <code>float</code> | required | Second reduced coordinate |

In [4]:
embeddings_result = run_minerva_embeddings(
    MinervaEmbeddingsInput(sequences=[SEQUENCE]),
    MinervaEmbeddingsConfig(
        model_checkpoint=CHECKPOINT, device=DEVICE, return_logits=True,
    ),
)
assert embeddings_result.success, embeddings_result.errors
embedding = embeddings_result.results[0]
print("Pooled dimension:", len(embedding.mean_embedding))
print("Token count:", len(embedding.tokens))
print("Logit columns:", embedding.vocab)
print("First tokens:", list(enumerate(embedding.tokens[:8], start=1)))

Starting worker


Running minerva


Loading gbrixi/minerva-mlm on cuda:0


Pooled dimension: 1280
Token count: 39
Logit columns: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'a', 'c', 'g', 't']
First tokens: [(1, '+'), (2, 'M'), (3, 'A'), (4, 'L'), (5, 'T'), (6, 'K'), (7, 'V'), (8, 'E')]


## Masked pseudo-log-likelihood

Every canonical protein/DNA position is masked in turn and scored from its context. Markers and ambiguous protein context are excluded from the score targets. Higher mean log-likelihood and lower perplexity indicate greater model predictability; they do not establish function.

In [5]:
display_api_reference("minerva-score", "input", "run_minerva_score")
display_api_reference("minerva-score", "config", "run_minerva_score")
display_api_reference("minerva-score", "output", "run_minerva_score")

**Input** — `MixedSequenceInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Prepared loci: uppercase protein, lowercase acgt, +/- strand markers (<+>/<-> accepted) |

**Config** — `MinervaScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device used for model inference |
| <code>model_checkpoint</code> | <code>Literal['gbrixi/minerva-mlm', 'gbrixi/minerva-mlm-8k']</code> | <code>'gbrixi/minerva-mlm'</code> | Minerva checkpoint; downloaded automatically from Hugging Face |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences or masked PLL variants per forward pass; reduce if memory is limited |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include masked biological logits; unscored context rows contain zeros |

**Output** — `MixedScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[MixedScoringMetrics]</code> | required | Per-locus scores in input order |

**Metrics** — `MixedScoringMetrics` (one set per `scores` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  | Sum over all positions. Grows with sequence length, so compare only at equal length. |
| <code>avg_log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  | Mean per position. Length independent, so comparable across sequences. |
| <code>perplexity</code> **(primary)** | <code>float</code> | <code>&gt;= 1</code> |  | Exponential of the negated mean log-likelihood. Lower means a more confident model. |

In [6]:
score_result = run_minerva_score(
    MinervaScoringInput(sequences=[SHORT_SEQUENCE]),
    MinervaScoringConfig(
        model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=1, return_logits=True,
    ),
)
assert score_result.success, score_result.errors
score = score_result.scores[0]
print("Mean log-likelihood:", score["avg_log_likelihood"])
print("Perplexity:", score["perplexity"])
print("Scored token positions:", score.scored_positions)

Starting worker


Running minerva


Loading gbrixi/minerva-mlm on cuda:0


Mean log-likelihood: -1.9868495035916567
Perplexity: 7.292522467001365
Scored token positions: [2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19]


## Sampling while preserving modality

The automatic strategy selects two canonical biological positions. `fixed_positions=[2]` protects the first amino acid because the initial strand marker occupies position 1. The model may sample the original token, so two selected sites need not mean two substitutions.

In [7]:
display_api_reference("minerva-sample", "input", "run_minerva_sample")
display_api_reference("minerva-sample", "config", "run_minerva_sample")
display_api_reference("minerva-sample", "output", "run_minerva_sample")

**Input** — `MixedSequenceSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Prepared mixed loci; '_' marks editable sites with explicit mask_modalities |
| <code>mask_modalities</code> | <code>list[dict[int, Literal['protein', 'dna']]] &#124; None</code> | <code>None</code> | Per-locus maps from 1-indexed masked token positions to protein or dna |

**Config** — `MinervaSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device used for model inference |
| <code>model_checkpoint</code> | <code>Literal['gbrixi/minerva-mlm', 'gbrixi/minerva-mlm-8k']</code> | <code>'gbrixi/minerva-mlm'</code> | Minerva checkpoint; downloaded automatically from Hugging Face |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences or masked PLL variants per forward pass; reduce if memory is limited |
| <code>masking_strategy</code> | <code>MaskingStrategy</code> | <code>MaskingStrategy(method='random', num_mutations=None, mask_fraction=None, fixed_positions=None, temperature=1.0)</code> | Select editable model-token positions; explicit '_' masks bypass selection |
| <code>sampling_method</code> | <code>Literal['single_pass', 'iterative_refinement']</code> | <code>'single_pass'</code> | Fill all masks once, or progressively commit predictions across refinement rounds |
| <code>temperature</code> | <code>float</code> | <code>1.0</code> | Sampling temperature within each position's protein or DNA alphabet |
| <code>top_p</code> | <code>float</code> | <code>1.0</code> | Iterative nucleus threshold; 1.0 disables nucleus filtering |
| <code>num_steps</code> | <code>int</code> | <code>20</code> | Number of iterative refinement rounds |
| <code>schedule</code> | <code>Literal['cosine', 'linear']</code> | <code>'cosine'</code> | Iterative unmask schedule |
| <code>strategy</code> | <code>Literal['random', 'entropy']</code> | <code>'random'</code> | Commit masked sites randomly or by lowest predictive entropy |
| <code>temperature_annealing</code> | <code>bool</code> | <code>True</code> | Cool sampling temperature over iterative refinement rounds |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include biological logits from a final forward pass over each completed sequence |

**Output** — `MixedSampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[MixedSequenceSample]</code> | required | Sampled locus bundles in input order |

**`MixedSequenceSample`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | Completed mixed protein/DNA locus with strand markers |
| <code>logits</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Optional per-token biological logits with shape (L, 24) |
| <code>tokens</code> | <code>list[str]</code> | required | Completed model-token sequence, including strand markers |
| <code>vocab</code> | <code>list[str]</code> | <code>['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'a', 'c', 'g', 't']</code> | Biological-logit column order |

In [8]:
sample_result = run_minerva_sample(
    MinervaSampleInput(sequences=[SEQUENCE]),
    MinervaSampleConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        masking_strategy=MaskingStrategy(num_mutations=2, fixed_positions=[2]),
        temperature=1.0,
        seed=7,
    ),
)
assert sample_result.success, sample_result.errors
sample = sample_result.results[0]
assert len(sample.tokens) == len(embedding.tokens)
assert sample.tokens[1] == embedding.tokens[1]
print("Original:", SEQUENCE)
print("Sampled: ", sample.sequence)

Starting worker


Running minerva


Loading gbrixi/minerva-mlm on cuda:0


Original: +MALTKVEKRNRIK+aatttaaggaa-MLGIDNIERVKP
Sampled:  +MALRKVEKRNRIK+aatttaaggaa-MLGIINIERVKP


### Explicit masks with a protein/DNA label per site

An underscore does not carry case, so each premasked position needs an explicit modality. The mapping uses 1-indexed token positions. This example fills one protein and one DNA site through four refinement rounds. Explicit masks bypass automatic position selection.

In [9]:
masked_tokens = list(embedding.tokens)
protein_position = 2
dna_position = next(i for i, token in enumerate(masked_tokens, start=1) if token in "acgt")
masked_tokens[protein_position - 1] = "_"
masked_tokens[dna_position - 1] = "_"

premasked_result = run_minerva_sample(
    MinervaSampleInput(
        sequences=["".join(masked_tokens)],
        mask_modalities=[{protein_position: "protein", dna_position: "dna"}],
    ),
    MinervaSampleConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        sampling_method="iterative_refinement",
        num_steps=4,
        top_p=0.9,
        temperature_annealing=False,
        seed=7,
    ),
)
assert premasked_result.success, premasked_result.errors
completed = premasked_result.results[0]
assert completed.tokens[protein_position - 1] in "ACDEFGHIKLMNPQRSTVWY"
assert completed.tokens[dna_position - 1] in "acgt"
print(completed.sequence)

Starting worker


Running minerva


Loading gbrixi/minerva-mlm on cuda:0


+MALTKVEKRNRIK+tatttaaggaa-MLGIDNIERVKP


## Relaxed-sequence gradient

The template fixes modality and strand markers. `one_hot_mixed_logits()` creates one 24-column row per token, with zero rows for fixed context. The default input temperature applies softmax within the allowed alphabet; use `temperature=None` only for already normalized modality-restricted probabilities. The model weights remain frozen.

In [10]:
display_api_reference("minerva-gradient", "input", "run_minerva_gradient")
display_api_reference("minerva-gradient", "config", "run_minerva_gradient")
display_api_reference("minerva-gradient", "output", "run_minerva_gradient")

**Input** — `MixedSequenceGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | Prepared template fixing each token's modality and strand markers |
| <code>logits</code> | <code>list[list[float]]</code> | required | Token-aligned (L, 24) state in ACDEFGHIKLMNPQRSTVWYacgt order; fixed rows are zero |
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Apply softmax(input / T); None requires modality-restricted probabilities |

**Config** — `MinervaGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device used for model inference |
| <code>model_checkpoint</code> | <code>Literal['gbrixi/minerva-mlm', 'gbrixi/minerva-mlm-8k']</code> | <code>'gbrixi/minerva-mlm'</code> | Minerva checkpoint; downloaded automatically from Hugging Face |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences or masked PLL variants per forward pass; reduce if memory is limited |
| <code>use_ste</code> | <code>bool</code> | <code>False</code> | Use hard one-hot forward tokens with soft-probability gradients |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward and return the gradient; False evaluates only the masked PLL objective |

**Output** — `MixedGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gradient</code> | <code>list[list[float]] &#124; list[list[list[float]]] &#124; None</code> | <code>None</code> | Gradient of the loss with respect to the input logits; matches input shape and column order |
| <code>loss</code> | <code>float</code> | required | Scalar objective for this sequence; lower is better (typically a positive negative log-likelihood) |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Auxiliary metrics reported alongside the scalar objective value. |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |
| <code>tokens</code> | <code>list[str]</code> | required | Model-token axis shared by input logits and returned gradient |

In [11]:
gradient_result = run_minerva_gradient(
    MinervaGradientInput(
        sequence=SHORT_SEQUENCE,
        logits=one_hot_mixed_logits(SHORT_SEQUENCE, sharpness=2.0),
        temperature=1.0,
    ),
    MinervaGradientConfig(model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=1),
)
assert gradient_result.success, gradient_result.errors
assert gradient_result.gradient is not None
print("Masked mean NLL:", gradient_result.loss)
print("Gradient shape:", (len(gradient_result.gradient), len(gradient_result.vocab)))
for token, row in zip(gradient_result.tokens, gradient_result.gradient, strict=True):
    if token in ("+", "-"):
        assert all(value == 0.0 for value in row)

Starting worker


Running minerva


Loading gbrixi/minerva-mlm on cuda:0


Masked mean NLL: 2.0352424047887325
Gradient shape: (19, 24)


## Interaction maps

Each selected head returns its raw `(L, L)` probability matrix, with labels for both axes. Markers are included and padding is removed. No symmetry, threshold, diagonal removal, or strand averaging is imposed. All three heads are requested here; select fewer to reduce output size. The default two-layer heads can be changed with `interaction_layers=6`.

These are model interaction signals, not an experimentally established contact map or an RNA secondary-structure call.

In [12]:
display_api_reference("minerva-interactions", "input", "run_minerva_interactions")
display_api_reference("minerva-interactions", "config", "run_minerva_interactions")
display_api_reference("minerva-interactions", "output", "run_minerva_interactions")

**Input** — `MixedSequenceInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Prepared loci: uppercase protein, lowercase acgt, +/- strand markers (<+>/<-> accepted) |

**Config** — `MinervaInteractionsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device used for model inference |
| <code>model_checkpoint</code> | <code>Literal['gbrixi/minerva-mlm', 'gbrixi/minerva-mlm-8k']</code> | <code>'gbrixi/minerva-mlm'</code> | Minerva checkpoint; downloaded automatically from Hugging Face |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences or masked PLL variants per forward pass; reduce if memory is limited |
| <code>heads</code> | <code>list[Literal['base_pairing', 'protein', 'repeat']]</code> | <code>['base_pairing', 'protein', 'repeat']</code> | Named interaction heads to return; select fewer to reduce output size |
| <code>interaction_layers</code> | <code>Literal[2, 6]</code> | <code>2</code> | Use interaction heads trained on the last two or six transformer layers |

**Output** — `SequenceInteractionsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[SequenceInteractions]</code> | required | Interaction bundles in input order |

**`SequenceInteractions`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>maps</code> | <code>dict[str, SequenceInteractionMap]</code> | required | Named interaction probability maps |

**`SequenceInteractionMap`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>tokens</code> | <code>list[str]</code> | required | Ordered labels for both matrix axes |
| <code>values</code> | <code>list[list[float]]</code> | required | Square token-by-token interaction probability matrix |

In [13]:
interactions_result = run_minerva_interactions(
    MinervaInteractionsInput(sequences=[SEQUENCE]),
    MinervaInteractionsConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        heads=["base_pairing", "protein", "repeat"],
        interaction_layers=2,
    ),
)
assert interactions_result.success, interactions_result.errors
for head, interaction_map in interactions_result.results[0].maps.items():
    length = len(interaction_map.tokens)
    assert len(interaction_map.values) == length
    assert all(len(row) == length for row in interaction_map.values)
    print(head, (length, length), interaction_map.tokens[:5])

Starting worker


Running minerva


Loading gbrixi/minerva-mlm on cuda:0


base_pairing (39, 39) ['+', 'M', 'A', 'L', 'T']
protein (39, 39) ['+', 'M', 'A', 'L', 'T']
repeat (39, 39) ['+', 'M', 'A', 'L', 'T']


## Export

Exports go to the temporary directory created above. Embedding CSV contains pooled vectors; score, sample, and gradient JSON preserve their result metadata.

Interaction maps default to compressed NPZ. Matrix and string-token arrays can be read with `allow_pickle=False`; JSON is also supported. Dense maps grow quadratically with token count.

In [14]:
embeddings_result.export(name="minerva_embeddings", export_path=OUTPUT_DIR, file_format="csv")
score_result.export(name="minerva_scores", export_path=OUTPUT_DIR, file_format="json")
sample_result.export(name="minerva_samples", export_path=OUTPUT_DIR, file_format="json")
gradient_result.export(name="minerva_gradient", export_path=OUTPUT_DIR, file_format="json")
interactions_result.export(name="minerva_interactions", export_path=OUTPUT_DIR)
interactions_result.export(name="minerva_interactions", export_path=OUTPUT_DIR, file_format="json")

import numpy as np

with np.load(OUTPUT_DIR / "minerva_interactions.npz", allow_pickle=False) as archive:
    print("NPZ entries:", archive.files)
    print("Protein map shape:", archive["0_protein"].shape)
    print("Token labels:", archive["0_protein_tokens"][:5])
print("Written files:", sorted(path.name for path in OUTPUT_DIR.iterdir()))

NPZ entries: ['0_base_pairing', '0_base_pairing_tokens', '0_protein', '0_protein_tokens', '0_repeat', '0_repeat_tokens']
Protein map shape: (39, 39)
Token labels: ['+' 'M' 'A' 'L' 'T']
Written files: ['minerva_embeddings.csv', 'minerva_gradient.json', 'minerva_interactions.json', 'minerva_interactions.npz', 'minerva_samples.json', 'minerva_scores.json']
